# 02 — Construcción del dataset bilateral actualizado (Sudamérica)

Este notebook construye el panel bilateral de **migración y movilidad** para Sudamérica combinando tres fuentes y codificando todos los regímenes de libre movimiento (FMR) desde cero.

| Variable | Fuente |
|---|---|
| `mstock` | Guy Abel 2025 (`stock_mean`) |
| `mflow` | Guy Abel 2025 (`mig_prev`) |
| `vflow_new` | GTMD2 (`gtmd2_vflow_int`) para la mayoría de corredores |
| `vflow_new` (j=ARG) | `turismo_anio_ARG.csv` cuando hay dato; si no, UNWTO fallback de GTMD2 |
| FMR dummies | Codificados directamente — sin depender de `did_latam_data.csv` |

El dataset resultante cubre los **13 países de Sudamérica** (10 con FMR + GUY/SUR/GUF como controles), años 1995–2022.

---

## Lógica de `vflow_new`

```
Para todos los corredores:
  vflow_new = gtmd2_vflow_int

Para corredores con destino = ARG:
  1. Si turismo_anio_ARG tiene dato para (origen, año) → usar ese valor
  2. Si no → probar unwto_tflow_112, luego 111, luego 122, luego 121
  3. Si ninguno → NaN

# Para agregar otro país en el futuro: añadir entrada en VFLOW_OVERRIDES (Sección 4)
```

In [ ]:
import pandas as pd
import numpy as np
import os

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 150)

In [ ]:
# ── Rutas ────────────────────────────────────────────────────────────────────
GTMD2_PATH    = "C:/Data/GTMD2/GTMD2_Data_MIGMOBS_share.csv"
ABEL_PATH     = "C:/Data/Guy_abel/mig_bilateral_abel_2025.csv"
ARG_TOUR_PATH = "Data/turismo_anio_ARG.csv"
OUTPUT_PATH   = "Output/did_latam_data_v2.csv"

# ── Países de Sudamérica ──────────────────────────────────────────────────────
# 10 países con participación en algún FMR + 3 controles sin FMR (necesarios para FEs del PPML)
SA_COUNTRIES = [
    "ARG", "BOL", "BRA", "CHL", "COL", "ECU",
    "PRY", "PER", "URY", "VEN",
    "GUY", "SUR", "GUF",
]

# ── Años ─────────────────────────────────────────────────────────────────────
YEAR_MIN = 1995
YEAR_MAX = 2022

# ── Columnas a cargar de GTMD2 (1.4 GB — minimizar memoria) ──────────────────
GTMD2_COLS = [
    'code_i', 'code_j', 'iso3code_i', 'iso3code_j', 'year',
    'country_i', 'country_j',
    'gtmd2_vflow_int',
    'unwto_tflow_112', 'unwto_tflow_111',
    'unwto_tflow_122', 'unwto_tflow_121',
]

# Orden de prioridad para fallback UNWTO (cuando no hay override disponible)
UNWTO_PRIORITY = ['unwto_tflow_112', 'unwto_tflow_111', 'unwto_tflow_122', 'unwto_tflow_121']

# ── Mapeo pais_agrupado → iso3code (turismo_anio_ARG.csv) ────────────────────
ARG_TOUR_MAP = {
    'Bolivia':  'BOL',
    'Brasil':   'BRA',
    'Chile':    'CHL',
    'Paraguay': 'PRY',
    'Uruguay':  'URY',
}

---
## Sección 1 — Países de Sudamérica

La lista `SA_COUNTRIES` está definida en los parámetros de arriba. Incluye:
- **10 países con FMR**: ARG, BOL, BRA, CHL, COL, ECU, PRY, PER, URY, VEN
- **3 controles sin FMR**: GUY (Guyana), SUR (Surinam), GUF (Guayana Francesa) — nunca participaron en ningún régimen; son necesarios para identificar los efectos fijos en el PPML

In [ ]:
print(f"Países de Sudamérica ({len(SA_COUNTRIES)}):")
print(SA_COUNTRIES)

---
## Sección 2 — GTMD2

Se carga el archivo GTMD2 (1.4 GB, 158 columnas) cargando **solo las columnas necesarias** y filtrando inmediatamente a los corredores SA + rango de años. Esto reduce el uso de memoria de ~10 GB a unos pocos MB.

Las columnas que se cargan son:
- Identificadores: `code_i/j`, `iso3code_i/j`, `year`, `country_i/j`
- Variable principal de vflow: `gtmd2_vflow_int`
- Alternativas UNWTO para fallback Argentina: `unwto_tflow_112/111/122/121`

In [ ]:
print("Cargando GTMD2 (puede tardar 1-2 minutos)...")
gtmd2_raw = pd.read_csv(GTMD2_PATH, usecols=GTMD2_COLS, low_memory=False)
print(f"  Total filas: {len(gtmd2_raw):,}")

gtmd2 = gtmd2_raw[
    gtmd2_raw['iso3code_i'].isin(SA_COUNTRIES) &
    gtmd2_raw['iso3code_j'].isin(SA_COUNTRIES) &
    gtmd2_raw['year'].between(YEAR_MIN, YEAR_MAX)
].copy()
del gtmd2_raw  # liberar memoria

print(f"  Después de filtrar (SA + {YEAR_MIN}-{YEAR_MAX}): {len(gtmd2):,} filas")
print(f"  Corredores únicos: {gtmd2.groupby(['iso3code_i','iso3code_j']).ngroups}")
print(f"  Años: {sorted(gtmd2['year'].unique())}")
print(f"\n  Cobertura gtmd2_vflow_int:  {gtmd2['gtmd2_vflow_int'].notna().sum():,} filas no-NaN")
for col in UNWTO_PRIORITY:
    print(f"  Cobertura {col}: {gtmd2[col].notna().sum():,} filas no-NaN")

In [ ]:
# ── Diagnóstico: cobertura de gtmd2_vflow_int por corredor (sin GUY/SUR/GUF) ──
EXCL = {"GUY", "SUR", "GUF"}
SA10 = [c for c in SA_COUNTRIES if c not in EXCL]

gtmd2_10 = gtmd2[
    gtmd2['iso3code_i'].isin(SA10) &
    gtmd2['iso3code_j'].isin(SA10) &
    (gtmd2['iso3code_i'] != gtmd2['iso3code_j'])
].copy()

all_years = list(range(YEAR_MIN, YEAR_MAX + 1))
n_years   = len(all_years)  # 28

# ── Para cada (i, j, year) con gtmd2_vflow_int NaN: ¿hay algo en UNWTO? ──────
nan_rows = gtmd2_10[gtmd2_10['gtmd2_vflow_int'].isna()].copy()
for col in UNWTO_PRIORITY:
    nan_rows[f'has_{col}'] = nan_rows[col].notna()
nan_rows['any_unwto'] = nan_rows[[f'has_{c}' for c in UNWTO_PRIORITY]].any(axis=1)

# ── Tabla resumen por corredor ─────────────────────────────────────────────────
cov = (
    gtmd2_10
    .assign(has_data=gtmd2_10['gtmd2_vflow_int'].notna())
    .pivot_table(index=['iso3code_i', 'iso3code_j'], columns='year',
                 values='has_data', aggfunc='max')
    .reindex(columns=all_years)
    .fillna(False)
)

cov_summary = pd.DataFrame({
    'n_con_dato': cov.sum(axis=1).astype(int),
    'n_nan':      (n_years - cov.sum(axis=1)).astype(int),
}, index=cov.index)
cov_summary['pct'] = (100 * cov_summary['n_con_dato'] / n_years).round(1)

# Agregar: de los NaN, cuántos tienen al menos un UNWTO
unwto_by_corr = (
    nan_rows.groupby(['iso3code_i', 'iso3code_j'])['any_unwto']
    .agg(n_unwto_disponible='sum', n_nan_total='count')
)
cov_summary = cov_summary.join(unwto_by_corr, how='left')
cov_summary['n_unwto_disponible'] = cov_summary['n_unwto_disponible'].fillna(0).astype(int)
cov_summary['n_nan_total']        = cov_summary['n_nan_total'].fillna(0).astype(int)

sin_dato = cov_summary[cov_summary['n_con_dato'] == 0]
parcial  = cov_summary[(cov_summary['n_con_dato'] > 0) & (cov_summary['n_con_dato'] < n_years)]
completo = cov_summary[cov_summary['n_con_dato'] == n_years]

print(f"Diagnóstico gtmd2_vflow_int — {len(SA10)} países SA (sin GUY/SUR/GUF)")
print(f"  Corredores analizados  : {len(cov_summary)}")
print(f"  Sin ningún dato        : {len(sin_dato)}")
print(f"  Datos parciales        : {len(parcial)}")
print(f"  Cobertura completa     : {len(completo)}")

if len(sin_dato):
    print(f"\n─── Corredores SIN dato en gtmd2_vflow_int (0/{n_years} años) ───")
    display(sin_dato.sort_index())

if len(parcial):
    print(f"\n─── Corredores con datos PARCIALES ───")
    display(parcial.sort_values('n_con_dato'))

# ── Detalle año a año de los periodos faltantes (sin UNWTO disponible) ─────────
problematicos = cov_summary[cov_summary['n_nan_total'] > 0].index
if len(problematicos):
    print(f"\n─── Años faltantes por corredor (gtmd2=NaN) + UNWTO disponible ───")
    rows_detail = []
    for (i, j) in problematicos:
        sub = gtmd2_10[
            (gtmd2_10['iso3code_i'] == i) & (gtmd2_10['iso3code_j'] == j) &
            gtmd2_10['gtmd2_vflow_int'].isna()
        ][['year'] + UNWTO_PRIORITY].sort_values('year')
        sub.insert(0, 'iso3code_i', i)
        sub.insert(1, 'iso3code_j', j)
        rows_detail.append(sub)
    detail = pd.concat(rows_detail, ignore_index=True)
    detail['any_unwto'] = detail[UNWTO_PRIORITY].notna().any(axis=1)
    display(detail)

---
## Sección 3 — Guy Abel 2025

Se carga el dataset bilateral de Abel 2025 para obtener:
- `mstock` ← `stock_mean`: stock bilateral de migrantes
- `mflow` ← `mig_prev`: flujo migratorio bilateral

Abel usa `orig`/`dest` como nombres de columna (iso3codes), que se renombran a `iso3code_i`/`iso3code_j`.

In [ ]:
print("Cargando Abel 2025...")
abel_raw = pd.read_csv(ABEL_PATH, low_memory=False)
print(f"  Total filas: {len(abel_raw):,}")
print(f"  Columnas: {list(abel_raw.columns)}")

abel = abel_raw.rename(columns={'orig': 'iso3code_i', 'dest': 'iso3code_j'})
abel = abel[
    abel['iso3code_i'].isin(SA_COUNTRIES) &
    abel['iso3code_j'].isin(SA_COUNTRIES) &
    abel['year'].between(YEAR_MIN, YEAR_MAX)
][['iso3code_i', 'iso3code_j', 'year', 'stock_mean', 'mig_prev']].copy()
del abel_raw

abel = abel.rename(columns={'stock_mean': 'mstock', 'mig_prev': 'mflow'})

print(f"\n  Después de filtrar: {len(abel):,} filas")
print(f"  Corredores: {abel.groupby(['iso3code_i','iso3code_j']).ngroups}")
print(f"  mstock no-NaN: {abel['mstock'].notna().sum():,}")
print(f"  mflow  no-NaN: {abel['mflow'].notna().sum():,}")
print()
display(abel.describe())

---
## Sección 4 — Turismo Argentina (`turismo_anio_ARG.csv`)

El archivo contiene series de visitantes a Argentina por grupos de países (INDEC/MINTUR). Los grupos que se corresponden directamente con un país de SA son: **Bolivia, Brasil, Chile, Paraguay, Uruguay**. El resto (EE.UU. y Canadá, Europa, Resto de América, Resto del mundo) no tienen correspondencia directa y quedarán cubiertos por el fallback UNWTO de GTMD2.

La columna `Viajes` es el número de viajes de visitantes, que mapea a `vflow`.

In [ ]:
arg_tour_raw = pd.read_csv(ARG_TOUR_PATH)
print("Columnas:", list(arg_tour_raw.columns))
print("Grupos disponibles:", sorted(arg_tour_raw['pais_agrupado'].unique()))

# Filtrar solo grupos con correspondencia directa a iso3code SA
arg_tour = arg_tour_raw[arg_tour_raw['pais_agrupado'].isin(ARG_TOUR_MAP)].copy()
arg_tour['iso3code_i'] = arg_tour['pais_agrupado'].map(ARG_TOUR_MAP)
arg_tour['iso3code_j'] = 'ARG'
arg_tour = arg_tour.rename(columns={'anio': 'year', 'Viajes': 'vflow_override'})
arg_tour = arg_tour[['iso3code_i', 'iso3code_j', 'year', 'vflow_override']]
arg_tour = arg_tour[arg_tour['year'].between(YEAR_MIN, YEAR_MAX)].copy()

print(f"\nFilas útiles (dentro de {YEAR_MIN}-{YEAR_MAX}): {len(arg_tour)}")
print(f"Orígenes cubiertos: {sorted(arg_tour['iso3code_i'].unique())}")
print(f"NaN en vflow_override: {arg_tour['vflow_override'].isna().sum()}")
display(arg_tour.head(10))

---
## Sección 5 — Diccionario de overrides de `vflow` (extensible)

El sistema de overrides permite sustituir la fuente de `vflow` para cualquier país destino. Cada entrada en `VFLOW_OVERRIDES` es un DataFrame con columnas `[iso3code_i, iso3code_j, year, vflow_override]`.

**Para agregar un nuevo país en el futuro**, agregar una entrada al diccionario aquí y preparar el DataFrame correspondiente en una celda nueva arriba.

In [ ]:
# ── Diccionario de overrides ──────────────────────────────────────────────────
# Formato: {iso3code_j: DataFrame con [iso3code_i, iso3code_j, year, vflow_override]}
#
# Para agregar otro país:
#   1. Preparar un df_nuevo con esas columnas en una celda nueva
#   2. Agregar aquí:  VFLOW_OVERRIDES['XYZ'] = df_nuevo

VFLOW_OVERRIDES = {
    'ARG': arg_tour,   # turismo_anio_ARG.csv para BOL/BRA/CHL/PRY/URY → ARG
}

print("Overrides configurados:")
for dest, df_ov in VFLOW_OVERRIDES.items():
    cob = df_ov['vflow_override'].notna().sum()
    print(f"  dest={dest}  |  filas={len(df_ov)}  |  no-NaN={cob}"
          f"  |  orígenes={sorted(df_ov['iso3code_i'].unique())}")

---
## Sección 6 — Construcción del dataset base

Se parte de GTMD2 (que provee la estructura del panel y los identificadores) y se le agrega Abel 2025 para `mstock` y `mflow`.

In [ ]:
df = gtmd2.copy()
df['iso3code_ij'] = df['iso3code_i'] + '_' + df['iso3code_j']

# ── Merge con Abel ────────────────────────────────────────────────────────────
df = df.merge(abel, on=['iso3code_i', 'iso3code_j', 'year'], how='left')

print("Cobertura de Abel en el dataset SA:")
print(f"  Filas totales : {len(df):,}")
print(f"  mstock no-NaN : {df['mstock'].notna().sum():,} ({100*df['mstock'].notna().mean():.1f}%)")
print(f"  mflow  no-NaN : {df['mflow'].notna().sum():,} ({100*df['mflow'].notna().mean():.1f}%)")

# Corredores sin ningún dato de Abel
no_abel = df.groupby(['iso3code_i','iso3code_j'])[['mstock','mflow']].apply(
    lambda g: g['mstock'].isna().all()
)
print(f"\nCorredores sin datos de Abel: {no_abel.sum()}")
if no_abel.sum() > 0:
    print(no_abel[no_abel].index.tolist())

---
## Sección 7 — Construcción de `vflow_new`

Se aplica la lógica de fuentes en cascada:

| Corredor | Prioridad 1 | Prioridad 2–5 |
|---|---|---|
| Todos excepto j en OVERRIDES | `gtmd2_vflow_int` | — |
| j = ARG, i ∈ {BOL,BRA,CHL,PRY,URY} | `turismo_anio_ARG` (si hay dato) | `unwto_tflow_112 → 111 → 122 → 121` |
| j = ARG, otro origen | `gtmd2_vflow_int` | `unwto_tflow_112 → 111 → 122 → 121` |

La columna `vflow_new_source` registra qué fuente se usó en cada fila para trazabilidad.

In [ ]:
# ── Paso 1: default = gtmd2_vflow_int para todos ─────────────────────────────
df['vflow_new']        = df['gtmd2_vflow_int']
df['vflow_new_source'] = np.where(df['gtmd2_vflow_int'].notna(), 'gtmd2_vflow_int', None)

# ── Paso 2: aplicar overrides por país destino ────────────────────────────────
for dest_country, df_ov in VFLOW_OVERRIDES.items():

    idx_dest = df.index[df['iso3code_j'] == dest_country]

    # Lookup del override usando (iso3code_i, year) como clave
    ov_lookup = df_ov.set_index(['iso3code_i', 'year'])['vflow_override']

    tmp = df.loc[idx_dest, ['iso3code_i', 'year']].copy()
    tmp['vflow_override'] = [
        ov_lookup.get((i, y), np.nan)
        for i, y in zip(tmp['iso3code_i'], tmp['year'])
    ]

    # Donde el override tiene dato → usarlo
    has_ov = tmp['vflow_override'].notna()
    df.loc[idx_dest[has_ov], 'vflow_new']        = tmp.loc[has_ov, 'vflow_override'].values
    df.loc[idx_dest[has_ov], 'vflow_new_source'] = f'override_{dest_country.lower()}'

    # Donde NO hay override Y vflow_new sigue siendo NaN → fallback UNWTO
    for col in UNWTO_PRIORITY:
        needs_fill = idx_dest[
            ~has_ov.values & df.loc[idx_dest, 'vflow_new'].isna().values &
            df.loc[idx_dest, col].notna().values
        ]
        if len(needs_fill):
            df.loc[needs_fill, 'vflow_new']        = df.loc[needs_fill, col]
            df.loc[needs_fill, 'vflow_new_source'] = col

# ── Resumen de fuentes ────────────────────────────────────────────────────────
print("Distribución de fuentes en vflow_new:")
print(df['vflow_new_source'].value_counts(dropna=False).to_string())
print(f"\nvflow_new no-NaN: {df['vflow_new'].notna().sum():,} / {len(df):,}"
      f" ({100*df['vflow_new'].notna().mean():.1f}%)")

# Detalle para Argentina
df_arg = df[df['iso3code_j'] == 'ARG']
print(f"\nDetalle para j=ARG ({len(df_arg)} filas):")
print(df_arg['vflow_new_source'].value_counts(dropna=False).to_string())

---
## Sección 8 — Codificación de indicadores FMR

Se construyen las variables dummy de cada Régimen de Libre Movimiento (FMR) siguiendo: Acosta & van der Baaren (2024), *Free Movement Regimes Dataset*.

La convención es **desde la perspectiva del destino**: `dummy = 1` si el país `j` aplica el FMR a los nacionales del país `i` en ese año.

| Variable | Régimen | Vigencia |
|---|---|---|
| `mercosur1` | MERCOSUR — miembros fundadores (ARG, BRA, PRY, URY) | desde 2009 |
| `mercosur2` | MERCOSUR — estados asociados (BOL, CHL, PER, COL, ECU) | desde 2009/2011/2012/2014 |
| `argfmrs` | Acuerdos bilaterales de Argentina pre-MERCOSUR | varios (1999–) |
| `ecuven` | Ecuador – Venezuela | ECU→VEN: 2011–2020; VEN→ECU: desde 2011 |
| `braury` | Brasil – Uruguay (más amplio que MERCOSUR) | desde 2017 |
| `can1` | Comunidad Andina — Decisión 545 (trabajadores específicos) | 2003–2020 |
| `can2` | Comunidad Andina — Decisión 878 (comprehensivo) | desde 2021 |

Variables de tratamiento agregadas:
- `FMR_all`: cualquier FMR activo (todos los anteriores)
- `FMR_Mercosur`: solo bloque MERCOSUR (`mercosur1` + `mercosur2`)
- `FMR_Mercosur_CAN`: MERCOSUR + CAN2 comprehensivo (excluye CAN1 por ser limitado a trabajadores específicos)

In [ ]:
# ── MERCOSUR1: miembros fundadores desde 2009 ─────────────────────────────────
# ARG, BRA, PRY, URY — simétrico (cada miembro aplica a los demás)
members_m1 = ["ARG", "BRA", "PRY", "URY"]
df["mercosur1"] = 0
for country_j in members_m1:
    df.loc[
        (df["iso3code_j"] == country_j) &
        (df["iso3code_i"].isin(members_m1)) &
        (df["year"] >= 2009),
        "mercosur1"
    ] = 1

# ── MERCOSUR2: estados asociados ──────────────────────────────────────────────
# BOL/CHL desde 2009; PER desde 2011; COL desde 2012; ECU desde 2014
# Nota: VEN nunca ratificó el acuerdo → no se codifica
df["mercosur2"] = 0

# ARG, BRA, PRY, URY aplican a todos los adherentes según sus fechas
for country_j in ["ARG", "BRA", "PRY", "URY"]:
    rules = {"BOL": 2009, "CHL": 2009, "PER": 2011, "COL": 2012, "ECU": 2014}
    for country_i, year_start in rules.items():
        df.loc[
            (df["iso3code_j"] == country_j) &
            (df["iso3code_i"] == country_i) &
            (df["year"] >= year_start),
            "mercosur2"
        ] = 1

# Chile: aplica a los 4 miembros y a BOL, pero NO a PER, COL, ECU
df.loc[(df["iso3code_j"] == "CHL") & (df["iso3code_i"].isin(members_m1)) & (df["year"] >= 2009), "mercosur2"] = 1
df.loc[(df["iso3code_j"] == "CHL") & (df["iso3code_i"] == "BOL") & (df["year"] >= 2009), "mercosur2"] = 1

# Bolivia: aplica a todos según sus fechas de adhesión
df.loc[(df["iso3code_j"] == "BOL") & (df["iso3code_i"].isin(members_m1)) & (df["year"] >= 2009), "mercosur2"] = 1
df.loc[(df["iso3code_j"] == "BOL") & (df["iso3code_i"] == "CHL") & (df["year"] >= 2009), "mercosur2"] = 1
df.loc[(df["iso3code_j"] == "BOL") & (df["iso3code_i"] == "PER") & (df["year"] >= 2011), "mercosur2"] = 1
df.loc[(df["iso3code_j"] == "BOL") & (df["iso3code_i"] == "COL") & (df["year"] >= 2012), "mercosur2"] = 1
df.loc[(df["iso3code_j"] == "BOL") & (df["iso3code_i"] == "ECU") & (df["year"] >= 2014), "mercosur2"] = 1

# Perú: adhiere en 2011
rules_per = {**{c: 2011 for c in members_m1}, "BOL": 2011, "CHL": 2011, "COL": 2012, "ECU": 2014}
for country_i, year_start in rules_per.items():
    df.loc[
        (df["iso3code_j"] == "PER") & (df["iso3code_i"] == country_i) & (df["year"] >= year_start),
        "mercosur2"
    ] = 1

# Colombia: adhiere en 2012; deja de aplicar a CHL desde 2020
rules_col = {**{c: 2012 for c in members_m1}, "BOL": 2012, "CHL": 2012, "PER": 2012, "ECU": 2014}
for country_i, year_start in rules_col.items():
    df.loc[
        (df["iso3code_j"] == "COL") & (df["iso3code_i"] == country_i) & (df["year"] >= year_start),
        "mercosur2"
    ] = 1
df.loc[(df["iso3code_j"] == "COL") & (df["iso3code_i"] == "CHL") & (df["year"] > 2019), "mercosur2"] = 0

# Ecuador: adhiere en 2014
rules_ecu = {**{c: 2014 for c in members_m1}, "BOL": 2014, "CHL": 2014, "PER": 2014, "COL": 2014}
for country_i, year_start in rules_ecu.items():
    df.loc[
        (df["iso3code_j"] == "ECU") & (df["iso3code_i"] == country_i) & (df["year"] >= year_start),
        "mercosur2"
    ] = 1

# ── ARGFMRS: acuerdos bilaterales de Argentina pre-MERCOSUR ───────────────────
df["argfmrs"] = 0
# ARG-BOL: 1999-2016 (simétrico)
for j, i in [("ARG", "BOL"), ("BOL", "ARG")]:
    df.loc[(df["iso3code_j"] == j) & (df["iso3code_i"] == i) &
           (df["year"] >= 1999) & (df["year"] <= 2016), "argfmrs"] = 1
# ARG-PER: desde 2000 (simétrico, en vigor continuo)
for j, i in [("ARG", "PER"), ("PER", "ARG")]:
    df.loc[(df["iso3code_j"] == j) & (df["iso3code_i"] == i) & (df["year"] >= 2000), "argfmrs"] = 1
# ARG-BRA y ARG-URY: desde 2006 (anticipan el MERCOSUR ampliado)
for j, i in [("ARG", "BRA"), ("BRA", "ARG"), ("ARG", "URY"), ("URY", "ARG")]:
    df.loc[(df["iso3code_j"] == j) & (df["iso3code_i"] == i) & (df["year"] >= 2006), "argfmrs"] = 1
# ARG-PRY: firmado pero NUNCA ratificado → no se codifica

# ── ECUVEN: Ecuador – Venezuela ───────────────────────────────────────────────
# Asimétrico: ECU aplica hasta 2020; VEN aplica de forma continua desde 2011
df["ecuven"] = 0
df.loc[(df["iso3code_j"] == "ECU") & (df["iso3code_i"] == "VEN") &
       (df["year"] >= 2011) & (df["year"] <= 2020), "ecuven"] = 1
df.loc[(df["iso3code_j"] == "VEN") & (df["iso3code_i"] == "ECU") &
       (df["year"] >= 2011), "ecuven"] = 1

# ── BRAURY: Brasil – Uruguay (más amplio que MERCOSUR) ────────────────────────
# Firmado en 2013, en vigor desde 2017; simétrico
df["braury"] = 0
df.loc[(df["iso3code_j"] == "BRA") & (df["iso3code_i"] == "URY") & (df["year"] >= 2017), "braury"] = 1
df.loc[(df["iso3code_j"] == "URY") & (df["iso3code_i"] == "BRA") & (df["year"] >= 2017), "braury"] = 1

# ── CAN1: Comunidad Andina, Decisión 545 — trabajadores (2003-2020) ───────────
core_can = ["BOL", "COL", "ECU", "PER"]
df["can1"] = 0
for country_j in core_can:
    df.loc[
        (df["iso3code_j"] == country_j) &
        (df["iso3code_i"].isin(core_can)) &
        (df["year"] >= 2003) & (df["year"] <= 2020),
        "can1"
    ] = 1

# ── CAN2: Comunidad Andina, Decisión 878 — comprehensivo (desde 2021) ─────────
df["can2"] = 0
for country_j in core_can:
    df.loc[
        (df["iso3code_j"] == country_j) &
        (df["iso3code_i"].isin(core_can)) &
        (df["year"] >= 2021),
        "can2"
    ] = 1

# ── Variables de tratamiento agregadas ────────────────────────────────────────
cols_all      = ["mercosur1", "mercosur2", "argfmrs", "ecuven", "braury", "can1", "can2"]
cols_mercosur = ["mercosur1", "mercosur2"]
cols_mer_can  = ["mercosur1", "mercosur2", "can2"]

df["FMR_all"]          = (df[cols_all].sum(axis=1)      > 0).astype(int)
df["FMR_Mercosur"]     = (df[cols_mercosur].sum(axis=1) > 0).astype(int)
df["FMR_Mercosur_CAN"] = (df[cols_mer_can].sum(axis=1)  > 0).astype(int)

# ── Variables auxiliares ───────────────────────────────────────────────────────
df["code_ij"]           = df.groupby(["iso3code_i", "iso3code_j"]).ngroup() + 1
df["un_region_inter_i"] = 5   # todos son Sudamérica (región 5 de la ONU)
df["un_region_inter_j"] = 5

print("Distribución de escenarios FMR:")
print(pd.DataFrame({
    "FMR_all":          df["FMR_all"].value_counts(),
    "FMR_Mercosur":     df["FMR_Mercosur"].value_counts(),
    "FMR_Mercosur_CAN": df["FMR_Mercosur_CAN"].value_counts(),
}).sort_index())
print(f"\nCorredores-año activos:")
print(f"  FMR_all:          {df['FMR_all'].sum():,}")
print(f"  FMR_Mercosur:     {df['FMR_Mercosur'].sum():,}")
print(f"  FMR_Mercosur_CAN: {df['FMR_Mercosur_CAN'].sum():,}")

In [ ]:
# ── Resumen: corredores y observaciones tratadas por FMR ─────────────────────
fmr_vars = ["FMR_all", "FMR_Mercosur", "FMR_Mercosur_CAN",
            "mercosur1", "mercosur2", "argfmrs", "can1", "can2", "ecuven", "braury"]

# Muestras por outcome
df_mig = df[df["mflow"].notna()]
df_vis = df[df["vflow_new"].notna()]

total_corr_m = df_mig.groupby(["iso3code_i", "iso3code_j"]).ngroups
total_corr_v = df_vis.groupby(["iso3code_i", "iso3code_j"]).ngroups
total_obs_m  = len(df_mig)
total_obs_v  = len(df_vis)

rows = []
for fmr in fmr_vars:
    treat_m       = df_mig[df_mig[fmr] == 1]
    corr_treat_m  = df_mig.groupby("code_ij")[fmr].max()
    treat_v       = df_vis[df_vis[fmr] == 1]
    corr_treat_v  = df_vis.groupby("code_ij")[fmr].max()
    rows.append({
        "FMR":                   fmr,
        "Corr. total (mig)":     total_corr_m,
        "Corr. tratados (mig)":  int((corr_treat_m == 1).sum()),
        "Obs tratadas (mig)":    len(treat_m),
        "% tratadas (mig)":      f"{100*len(treat_m)/total_obs_m:.1f}%",
        "Corr. total (vis)":     total_corr_v,
        "Corr. tratados (vis)":  int((corr_treat_v == 1).sum()),
        "Obs tratadas (vis)":    len(treat_v),
        "% tratadas (vis)":      f"{100*len(treat_v)/total_obs_v:.1f}%",
    })

summary_fmr = pd.DataFrame(rows).set_index("FMR")

print(f"Muestra migración  (mflow no-NaN): {total_obs_m:,} obs  |  {total_corr_m} corredores")
print(f"Muestra visitantes (vflow_new no-NaN): {total_obs_v:,} obs  |  {total_corr_v} corredores")
print()
display(summary_fmr)

---
## Sección 9 — Validación

Chequeos básicos de consistencia antes de exportar:
1. Número de filas y corredores vs. dataset de referencia
2. Duplicados
3. Cobertura de las variables principales
4. Comparación de `vflow_new` vs. `vflow` original para Argentina

In [ ]:
print("=" * 55)
print("VALIDACIÓN DEL DATASET")
print("=" * 55)

# 1. Tamaño y estructura
print(f"\n1. TAMAÑO")
print(f"  Filas totales        : {len(df):,}")
print(f"  Corredores únicos    : {df.groupby(['iso3code_i','iso3code_j']).ngroups}")
print(f"  Años cubiertos       : {df['year'].min()} – {df['year'].max()}")

# 2. Duplicados
dups = df.duplicated(subset=['iso3code_i', 'iso3code_j', 'year']).sum()
print(f"\n2. DUPLICADOS (iso3code_i, iso3code_j, year): {dups}")

# 3. Cobertura de variables principales
print(f"\n3. COBERTURA DE VARIABLES PRINCIPALES")
for col in ['mstock', 'mflow', 'vflow_new', 'gtmd2_vflow_int']:
    if col in df.columns:
        nn  = df[col].notna().sum()
        pct = 100 * nn / len(df)
        print(f"  {col:<22} {nn:>6,} no-NaN  ({pct:.1f}%)")

# 4. Cobertura FMR
print(f"\n4. COBERTURA FMR")
for col in ['FMR_all', 'FMR_Mercosur', 'mercosur1', 'mercosur2', 'argfmrs',
            'ecuven', 'braury', 'can1', 'can2']:
    if col in df.columns:
        nn = df[col].sum()
        print(f"  {col:<22} {nn:>6,} obs tratadas  ({100*nn/len(df):.1f}%)")

# 5. Detalle Argentina — fuentes de vflow_new
print(f"\n5. ARGENTINA — distribución de fuentes de vflow_new")
df_arg = df[df['iso3code_j'] == 'ARG']
print(f"  Filas j=ARG: {len(df_arg)}")
print(df_arg['vflow_new_source'].value_counts(dropna=False).to_string())
print()

# 6. Muestra de filas finales
display(df[['iso3code_i', 'iso3code_j', 'year', 'mflow', 'mstock',
            'vflow_new', 'vflow_new_source', 'FMR_all']].head(8))

---
## Sección 10 — Exportar dataset

Se seleccionan las columnas finales y se guarda en `Output/did_latam_data_v2.csv`.

In [ ]:
FINAL_COLS = [
    'code_i', 'code_j', 'iso3code_i', 'iso3code_j', 'iso3code_ij',
    'country_i', 'country_j',
    'year',
    'mflow', 'mstock',
    'vflow_new', 'vflow_new_source',
    'gtmd2_vflow_int',          # vflow original GTMD2 (para comparación)
    'mercosur1', 'mercosur2', 'argfmrs', 'ecuven', 'braury', 'can1', 'can2',
    'FMR_all', 'FMR_Mercosur', 'FMR_Mercosur_CAN',
    'code_ij',
    'un_region_inter_i', 'un_region_inter_j',
]

final_cols_ok = [c for c in FINAL_COLS if c in df.columns]
df_out = df[final_cols_ok].sort_values(['iso3code_i', 'iso3code_j', 'year']).reset_index(drop=True)

df_out.to_csv(OUTPUT_PATH, index=False)

print(f"Exportado: {OUTPUT_PATH}")
print(f"  Filas: {len(df_out):,}")
print(f"  Columnas: {len(df_out.columns)}")
print(f"\nColumnas finales:")
print(final_cols_ok)
print()
display(df_out.head(8))